# 中心极限定理完整教程：从模拟实验到实际应用

## 📚 教学目标
1. 理解中心极限定理的表述
2. 用模拟实验验证：不同总体分布下样本均值趋近正态
3. 理解样本量对抽样分布形态的影响
4. 掌握标准误公式：$\sigma_{\bar{x}} = \frac{\sigma}{\sqrt{n}}$
5. 理解中心极限定理在实际应用中的重要性

**参考书**：《基础统计学(第14版)》（Triola）第 6-4 节
**教学策略**：用四种不同形状的总体分布做模拟实验，直观展示 CLT 的"魔力"

## 1. 场景设定：为什么正态分布无处不在？

### 🎯 问题
人的身高、考试分数、股票波动、年平均气温……为什么这么多数据都近似正态分布？

答案就是**中心极限定理**（Central Limit Theorem, CLT）——统计学中最深刻、最实用的定理之一。

### 📖 书中的定义
> 给定任意分布的总体，当样本量足够大时（n ≥ 30），样本均值 $\bar{x}$ 的分布可以近似于正态分布。

### 📐 数学表述
如果总体均值为 $\mu$，标准差为 $\sigma$，则样本量为 n 的样本均值 $\bar{x}$ 满足：
$$\mu_{\bar{x}} = \mu$$
$$\sigma_{\bar{x}} = \frac{\sigma}{\sqrt{n}}$$
当 n 足够大时，$\bar{x} \sim N(\mu_{\bar{x}}, \sigma_{\bar{x}}^2)$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 设置中文字体和样式
import matplotlib.font_manager as fm
_cn_fonts = [f.name for f in fm.fontManager.ttflist if any(kw in f.name for kw in ['Hei', 'Song', 'PingFang', 'Arial Unicode', 'Noto Sans CJK', 'SimHei', 'Microsoft YaHei'])]
plt.rcParams['font.sans-serif'] = _cn_fonts[:3] + ['DejaVu Sans'] if _cn_fonts else ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style('whitegrid')

np.random.seed(42)
print("✅ 库导入完成")

## 2. CLT 验证实验设计

### 🔬 实验方案
我们将用**四种不同形状**的总体分布来验证 CLT：

| 总体分布 | 形状 | 特点 |
|---------|------|------|
| 均匀分布 | 平坦 | 完全对称，但不是钟形 |
| 指数分布 | 右偏 | 严重偏态 |
| 二项分布 | 近似对称 | 离散分布 |
| 正态分布 | 钟形 | 已经是正态 |

对每种总体，分别抽取 n = 2, 5, 10, 30 的样本，重复 1 万次，观察样本均值的分布。

In [ ]:
# ========== 步骤 1: 定义四种总体分布 ==========

distributions = {
    'Uniform(0,1)': {
        'generator': lambda size: np.random.uniform(0, 1, size),
        'mu': 0.5, 'sigma': 1/np.sqrt(12),
        'desc': '均匀分布：平坦，完全对称但非钟形'
    },
    'Exponential(λ=1)': {
        'generator': lambda size: np.random.exponential(1, size),
        'mu': 1.0, 'sigma': 1.0,
        'desc': '指数分布：严重右偏'
    },
    'Binomial(10,0.5)': {
        'generator': lambda size: np.random.binomial(10, 0.5, size),
        'mu': 5.0, 'sigma': np.sqrt(10*0.5*0.5),
        'desc': '二项分布：离散，近似对称'
    },
    'Normal(0,1)': {
        'generator': lambda size: np.random.normal(0, 1, size),
        'mu': 0.0, 'sigma': 1.0,
        'desc': '正态分布：已经是钟形'
    }
}

print(f"\n📊 步骤 1: 四种总体分布的参数")
print(f"{'分布':<20} {'μ':>8} {'σ':>8} {'描述'}")
print("-" * 70)
for name, params in distributions.items():
    print(f"{name:<20} {params['mu']:>8.4f} {params['sigma']:>8.4f} {params['desc']}")

In [ ]:
# ========== 步骤 2: 可视化四种总体的原始分布 ==========

fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, (name, params) in zip(axes, distributions.items()):
    data = params['generator'](10000)
    ax.hist(data, bins=50, density=True, alpha=0.6, color='steelblue', edgecolor='white')
    ax.axvline(x=params['mu'], color='red', linestyle='--', linewidth=2, label=f'μ={params["mu"]}')
    ax.set_xlabel('Value', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f'{name}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle('Four Population Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  四种总体分布的形状差异很大：")
print(f"  均匀分布平坦，指数分布右偏，二项分布离散，正态分布钟形")
print(f"  ➡ 接下来我们将看到：无论总体形状如何，样本均值的分布都会趋近正态！")

## 3. CLT 模拟实验：不同样本量的效果

### 🔬 实验步骤
对每种总体分布，分别用 n = 2, 5, 10, 30 的样本量，重复 1 万次抽样，绘制样本均值的直方图。

In [ ]:
# ========== 步骤 3: CLT 模拟实验 ==========

sample_sizes = [2, 5, 10, 30]
n_simulations = 10000

fig, axes = plt.subplots(4, 4, figsize=(20, 16))

for row, (dist_name, params) in enumerate(distributions.items()):
    for col, n_s in enumerate(sample_sizes):
        ax = axes[row, col]
        
        # 模拟抽样
        sample_means = np.array([
            np.mean(params['generator'](n_s))
            for _ in range(n_simulations)
        ])
        
        # 理论标准误
        se = params['sigma'] / np.sqrt(n_s)
        
        # 直方图
        ax.hist(sample_means, bins=40, density=True, alpha=0.6, color='steelblue', edgecolor='white')
        
        # 理论正态曲线
        x_min, x_max = np.min(sample_means), np.max(sample_means)
        x = np.linspace(x_min, x_max, 100)
        y = stats.norm.pdf(x, params['mu'], se)
        ax.plot(x, y, 'r-', linewidth=2)
        
        ax.axvline(x=params['mu'], color='#2ecc71', linestyle='--', linewidth=1.5)
        ax.set_xlabel('', fontsize=9)
        ax.set_ylabel('Density', fontsize=9)
        
        if row == 0:
            ax.set_title(f'n = {n_s}', fontsize=14, fontweight='bold')
        if col == 0:
            ax.set_ylabel(f'{dist_name}\nDensity', fontsize=10)
        
        ax.grid(alpha=0.3)

plt.suptitle('Central Limit Theorem: Effect of Sample Size', fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  每一行代表一种总体分布，每一列代表一种样本量")
print(f"  红色曲线是理论正态分布（均值=μ，标准差=σ/√n）")
print(f"  ➡ 关键发现：n ≥ 30 时，无论总体形状如何，样本均值分布都近似正态！")

## 4. 偏度随样本量的变化

### 📐 偏度的含义
偏度（Skewness）衡量分布的不对称程度。正态分布的偏度为 0。

CLT 告诉我们：随着 n 增大，样本均值分布的偏度趋于 0。

In [ ]:
# ========== 步骤 4: 偏度随样本量的变化 ==========

sample_sizes_skew = [1, 2, 5, 10, 20, 30, 50, 100]
n_sim_skew = 10000

print(f"\n📊 步骤 4: 样本均值分布的偏度 vs 样本量")
print(f"  （基于 {n_sim_skew} 次模拟实验）")
print(f"\n{'总体分布':<20} | {'n=1':>6} {'n=2':>6} {'n=5':>6} {'n=10':>6} {'n=30':>6} {'n=100':>6}")
print("-" * 75)

skewness_data = {}
for dist_name, params in distributions.items():
    skews = []
    for n_s in sample_sizes_skew:
        means = np.array([
            np.mean(params['generator'](n_s))
            for _ in range(n_sim_skew)
        ])
        skews.append(stats.skew(means))
    skewness_data[dist_name] = skews
    
    # 打印部分结果
    selected = [0, 1, 2, 3, 5, 7]  # n=1,2,5,10,30,100
    vals = ' '.join([f'{skews[i]:>6.3f}' for i in selected])
    print(f"{dist_name:<20} | {vals}")

print(f"\n💡 结论：")
print(f"  指数分布（原始偏度≈2）的样本均值偏度随 n 增大迅速趋于 0")
print(f"  正态分布（原始偏度≈0）的样本均值偏度始终接近 0")
print(f"  ➡ n ≥ 30 时，所有分布的偏度都接近 0（近似正态）")

In [ ]:
# ========== 可视化：偏度随样本量的变化 ==========

fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#3498db', '#e74c3c', '#2ecc71', '#e67e22']
markers = ['o', 's', '^', 'D']

for (dist_name, skews), color, marker in zip(skewness_data.items(), colors, markers):
    ax.plot(sample_sizes_skew, skews, color=color, marker=marker, linewidth=2, 
            markersize=8, label=dist_name)

ax.axhline(y=0, color='black', linestyle='-', linewidth=1, alpha=0.3)
ax.axhline(y=0.1, color='gray', linestyle='--', linewidth=1, alpha=0.5)
ax.axhline(y=-0.1, color='gray', linestyle='--', linewidth=1, alpha=0.5)

ax.set_xlabel('Sample Size n', fontsize=12)
ax.set_ylabel('Skewness of Sampling Distribution', fontsize=12)
ax.set_title('Skewness vs Sample Size for Different Populations', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  偏度接近 0 表示分布近似对称（正态）")
print(f"  指数分布（偏度最高）需要更大的 n 才能接近正态")
print(f"  ➡ 经验法则：n ≥ 30 对大多数分布已经足够")

## 5. 实际应用：波音 737 座椅宽度问题

### 📖 书中的例题
> 美国航空公司波音 737 飞机主舱有 126 个座位。成年男性的臀部宽度服从均值为 14.3 英寸，标准差为 0.9 英寸的正态分布。
> 1. 随机选取一个成年男性，求其臀宽大于 16.0 英寸的概率
> 2. 假设 126 个座位全被男性坐满，求他们平均臀宽大于 16.0 英寸的概率

这个问题完美展示了**单一值 vs 样本均值**的区别。

In [ ]:
# ========== 步骤 5: 波音 737 座椅宽度问题 ==========

# 总体参数
mu_hip = 14.3   # 均值（英寸）
sigma_hip = 0.9  # 标准差
x_value = 16.0   # 座椅宽度
n_passengers = 126

print(f"\n📊 步骤 5: 波音 737 座椅宽度问题")
print(f"  总体: 成年男性臀宽 ~ N(μ={mu_hip}, σ={sigma_hip})")
print(f"  座椅宽度 = {x_value} 英寸")
print(f"  座位数 n = {n_passengers}")

In [ ]:
# ========== 步骤 6: 单一值的概率 ==========

# 单一值的 z 分数
z_single = (x_value - mu_hip) / sigma_hip
prob_single = 1 - stats.norm.cdf(z_single)

print(f"\n📊 步骤 6: 单一成年男性臀宽 > {x_value} 英寸的概率")
print(f"  z = (x - μ) / σ = ({x_value} - {mu_hip}) / {sigma_hip} = {z_single:.2f}")
print(f"  P(X > {x_value}) = P(Z > {z_single:.2f}) = {prob_single:.4f}")
print(f"  即约 {prob_single*100:.2f}% 的成年男性臀宽超过 {x_value} 英寸")
print(f"\n  💡 每个航班大约有 {n_passengers * prob_single:.0f} 人会坐不下！")

In [ ]:
# ========== 步骤 7: 样本均值的概率（应用 CLT） ==========

# 样本均值的标准误
se_hip = sigma_hip / np.sqrt(n_passengers)

# 样本均值的 z 分数
z_mean = (x_value - mu_hip) / se_hip
prob_mean = 1 - stats.norm.cdf(z_mean)

print(f"\n📊 步骤 7: 126 个男性的平均臀宽 > {x_value} 英寸的概率")
print(f"  标准误 σ_x̄ = σ / √n = {sigma_hip} / √{n_passengers} = {se_hip:.6f}")
print(f"  z = (x̄ - μ) / σ_x̄ = ({x_value} - {mu_hip}) / {se_hip:.6f} = {z_mean:.2f}")
print(f"  P(X̄ > {x_value}) = P(Z > {z_mean:.2f}) = {prob_mean:.10f}")
print(f"\n  💡 结论：")
print(f"    单一男性: P > {prob_single:.4f}（约 {prob_single*100:.1f}%，可能发生）")
print(f"    126人均值: P ≈ {prob_mean:.10f}（几乎不可能发生！）")
print(f"    ➡ 样本均值的波动远小于单一值！这就是 CLT 的威力！")

In [ ]:
# ========== 可视化：单一值 vs 样本均值 ==========

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左图：单一值的分布
ax1 = axes[0]
x1 = np.linspace(mu_hip - 4*sigma_hip, mu_hip + 4*sigma_hip, 1000)
y1 = stats.norm.pdf(x1, mu_hip, sigma_hip)
ax1.plot(x1, y1, 'b-', linewidth=2, label=f'N({mu_hip}, {sigma_hip})')
ax1.fill_between(x1, 0, y1, where=(x1 >= x_value), alpha=0.3, color='#e74c3c', 
                 label=f'P(X>{x_value})={prob_single:.4f}')
ax1.axvline(x=x_value, color='red', linestyle='--', linewidth=2)
ax1.axvline(x=mu_hip, color='#2ecc71', linestyle='--', linewidth=1.5, label=f'μ={mu_hip}')
ax1.set_xlabel('Hip Width (inches)', fontsize=12)
ax1.set_ylabel('Density', fontsize=12)
ax1.set_title('Single Individual', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# 右图：样本均值的分布
ax2 = axes[1]
x2 = np.linspace(mu_hip - 4*se_hip, mu_hip + 4*se_hip, 1000)
y2 = stats.norm.pdf(x2, mu_hip, se_hip)
ax2.plot(x2, y2, 'b-', linewidth=2, label=f'N({mu_hip}, {se_hip:.4f})')
ax2.fill_between(x2, 0, y2, where=(x2 >= x_value), alpha=0.3, color='#e74c3c', 
                 label=f'P(X̄>{x_value})≈0')
ax2.axvline(x=x_value, color='red', linestyle='--', linewidth=2)
ax2.axvline(x=mu_hip, color='#2ecc71', linestyle='--', linewidth=1.5, label=f'μ={mu_hip}')
ax2.set_xlabel('Mean Hip Width (inches)', fontsize=12)
ax2.set_ylabel('Density', fontsize=12)
ax2.set_title(f'Mean of 126 Individuals (SE={se_hip:.4f})', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.suptitle('CLT Application: Boeing 737 Seat Width', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  左图：单一男性的臀宽分布，16 英寸处有约 {prob_single*100:.1f}% 的面积")
print(f"  右图：126 人平均臀宽的分布，16 英寸处几乎没有面积")
print(f"  ➡ 设计座椅时应考虑单一值（左图），而非平均值（右图）！")

## 6. 实际应用：人体体温真的是 98.6°F 吗？

### 📖 书中的例题
> 假设人的平均体温是 98.6°F，总体标准差为 0.62°F。随机选取 106 名受试者，得到平均体温为 98.2°F。如果总体均值真的是 98.6°F，获得 98.2°F 或更低样本均值的概率是多少？

这是一个**假设检验**的思维雏形——用 CLT 来判断观测值是否"显著"。

In [ ]:
# ========== 步骤 8: 体温问题 ==========

# 参数
mu_temp = 98.6      # 假设的总体均值
sigma_temp = 0.62   # 总体标准差
n_temp = 106        # 样本量
xbar_temp = 98.2    # 观测到的样本均值

# 标准误
se_temp = sigma_temp / np.sqrt(n_temp)

# z 分数
z_temp = (xbar_temp - mu_temp) / se_temp

# 概率
prob_temp = stats.norm.cdf(z_temp)

print(f"\n📊 步骤 8: 人体体温问题")
print(f"  假设: μ = {mu_temp}°F")
print(f"  总体标准差: σ = {sigma_temp}°F")
print(f"  样本量: n = {n_temp}")
print(f"  观测到的样本均值: x̄ = {xbar_temp}°F")
print(f"\n  计算过程：")
print(f"    标准误 σ_x̄ = σ/√n = {sigma_temp}/√{n_temp} = {se_temp:.6f}")
print(f"    z = (x̄ - μ)/σ_x̄ = ({xbar_temp} - {mu_temp})/{se_temp:.6f} = {z_temp:.4f}")
print(f"    P(X̄ ≤ {xbar_temp}) = P(Z ≤ {z_temp:.4f}) = {prob_temp:.15f}")
print(f"\n  💡 结论：")
print(f"    如果总体均值真的是 {mu_temp}°F，")
print(f"    观测到 x̄ ≤ {xbar_temp}°F 的概率几乎为 0！")
print(f"  这说明 98.6°F 的\"常识\"很可能是错误的！")


In [ ]:
# ========== 可视化：体温问题 ==========

fig, ax = plt.subplots(figsize=(10, 6))

# 样本均值的分布
x_temp = np.linspace(mu_temp - 4*se_temp, mu_temp + 4*se_temp, 1000)
y_temp = stats.norm.pdf(x_temp, mu_temp, se_temp)

ax.plot(x_temp, y_temp, 'b-', linewidth=2, label=f'Sampling Dist: N({mu_temp}, {se_temp:.4f})')

# 左尾面积
x_left = np.linspace(mu_temp - 4*se_temp, xbar_temp, 1000)
y_left = stats.norm.pdf(x_left, mu_temp, se_temp)
ax.fill_between(x_left, 0, y_left, alpha=0.3, color='#e74c3c', 
                label=f'P(X̄≤{xbar_temp}) ≈ 0+')

ax.axvline(x=xbar_temp, color='red', linestyle='--', linewidth=2, label=f'Observed x̄ = {xbar_temp}')
ax.axvline(x=mu_temp, color='#2ecc71', linestyle='--', linewidth=2, label=f'Assumed μ = {mu_temp}')

ax.set_xlabel('Body Temperature (°F)', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Is Average Body Temperature Really 98.6°F?', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  如果总体均值真的是 98.6°F，样本均值为 98.2°F 的概率几乎为 0")
print(f"  这是一个\"稀有事件\"——要么我们碰到了极罕见的事件，要么假设本身就是错的")
print(f"  ➡ 马里兰大学的研究者因此得出结论：人的平均体温低于 98.6°F！")

## 7. CLT 的"模糊"版本

### 📖 书中的引用
> "受许多小而不相关的随机效应影响的数据近似正态分布。这就解释了为什么正态分布无处不在——股市波动、学生体重、年平均气温、高考分数——所有这些都是许多不同影响的结果。"
> ——Gonick & Smith, *The Cartoon Guide to Statistics*

### 💡 深层含义
人的身高是遗传、环境、营养、医疗、地理等因素的**总和**。根据 CLT，许多随机效应的总和趋于正态分布。这就是正态分布无处不在的原因！

In [ ]:
# ========== 步骤 9: 多个随机效应之和趋于正态 ==========

# 模拟：多个均匀分布之和
n_effects = [1, 2, 5, 10, 30]
n_sim_sum = 10000

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, n_e in zip(axes, n_effects):
    # n_e 个均匀分布之和
    sums = np.sum(np.random.uniform(0, 1, (n_sim_sum, n_e)), axis=1)
    
    ax.hist(sums, bins=40, density=True, alpha=0.6, color='steelblue', edgecolor='white')
    
    # 理论正态曲线
    mu_sum = n_e * 0.5
    sigma_sum = np.sqrt(n_e * 1/12)
    x = np.linspace(mu_sum - 4*sigma_sum, mu_sum + 4*sigma_sum, 100)
    y = stats.norm.pdf(x, mu_sum, sigma_sum)
    ax.plot(x, y, 'r-', linewidth=2)
    
    ax.set_xlabel('Sum', fontsize=10)
    ax.set_ylabel('Density', fontsize=10)
    ax.set_title(f'{n_e} Uniform(s) Summed', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)

plt.suptitle('Sum of Uniform Distributions → Normal (CLT)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n💡 图解说明：")
print(f"  单个均匀分布是平坦的（左图）")
print(f"  但多个均匀分布之和迅速变成钟形（正态）！")
print(f"  这就是为什么\"受许多小随机效应影响的数据近似正态\"！")


## 📌 核心概念回顾

### 📌 中心极限定理 (CLT)
- **定义**: 对于任意总体，当样本量 n 足够大时，样本均值 $\bar{x}$ 的分布近似正态
- **公式**: $\bar{x} \sim N(\mu, \frac{\sigma^2}{n})$
- **条件**: n ≥ 30 或总体本身正态
- **意义**: 解释了正态分布为何无处不在

### 📌 标准误 (SEM)
- **公式**: $\sigma_{\bar{x}} = \frac{\sigma}{\sqrt{n}}$
- **含义**: 样本均值的标准差，衡量均值的波动程度
- **应用**: 区分单一值和样本均值的概率计算

### 📌 单一值 vs 样本均值
- **单一值**: 使用 $z = \frac{x - \mu}{\sigma}$
- **样本均值**: 使用 $z = \frac{\bar{x} - \mu}{\sigma / \sqrt{n}}$
- **关键区别**: 样本均值的波动远小于单一值！

### 🔗 完整流程
```
总体(任意分布) → 抽取样本(n≥30) → 计算均值 → 重复 → 分布趋于正态
      ↓                ↓              ↓         ↓          ↓
   μ, σ            简单随机样本       x̄       模拟实验    N(μ, σ²/n)
```

## ❌ 常见误区

### ❌ 误区 1: CLT 说原始数据趋于正态
**✓ 正确理解**: CLT 说的是**样本均值的分布**趋于正态，不是原始数据！原始数据的分布形状取决于总体。

### ❌ 误区 2: n ≥ 30 是绝对的门槛
**✓ 正确理解**: n ≥ 30 是经验法则。对于极度偏态的分布（如指数分布），可能需要更大的 n。对于已经正态的总体，任何 n 都可以。

### ❌ 误区 3: CLT 只适用于均值
**✓ 正确理解**: CLT 的经典形式针对样本均值。但类似的结果也适用于样本比例（二项分布的正态近似）。样本方差的分布则不同（卡方分布）。

### ❌ 误区 4: 如果总体是正态的，CLT 就不重要了
**✓ 正确理解**: 如果总体正态，样本均值对**任何** n 都精确服从正态分布。CLT 的真正价值在于：即使总体不正态，大样本下均值仍然近似正态！

### ❌ 误区 5: 样本均值的概率和单一值的概率是一样的
**✓ 正确理解**: 样本均值的波动（标准误 = σ/√n）远小于单一值的波动（标准差 = σ）。因此，极端均值的概率远小于极端单一值的概率（见飞机座椅例子）。